# 3aransiya · resumable Colab T4 pilot

Train the GRU encoder–decoder and bidirectional GRU/CTC on the same development corpus. The code is embedded in this notebook. The locked test stays closed.

1. Put the prepared `doda-word-pairs-v1.json` in the `3aransiya` folder in your private Google Drive.
2. In Colab choose **Runtime → Change runtime type → T4 GPU**, then run all cells.
3. After a disconnect, select the same runtime, reconnect Drive, and rerun with the same run name and training settings. Completed candidates skip training. Run one session per run folder.

The default saves after each optimizer step. If Drive writes become a bottleneck, increase `SAVE_EVERY` and accept replay of unsaved steps. The two checkpoint slots survive typical interrupted writes. Drive can still lose writes it hasn't synchronized.

[Colab runtime and Drive limits](https://research.google.com/colaboratory/faq.html)


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/3aransiya")
RUN_NAME = "t4-pilot-v1"
CANDIDATES = ["gru_encoder_decoder_v1", "bidirectional_gru_ctc_v1"]
EPOCHS = 2
BATCH_SIZE = 128
SEED = 42
SAVE_EVERY = 1
STEPS_THIS_SESSION = None  # e.g. 100 to stop after 100 additional batches per candidate


In [ ]:
from base64 import b64decode
from hashlib import sha256
from io import BytesIO
from zipfile import ZipFile

import torch

if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0).split():
    raise RuntimeError("Select a T4 GPU in Colab Runtime → Change runtime type")
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))

EXPECTED_DATA = "842ba2460f0d4219cb90a41cca0cdc079d18fd1146d6756a9562833c9f2fbfbb"
LOCAL = Path("/content/3aransiya-pilot")
LOCAL.mkdir(exist_ok=True)
WORD_PAIRS = LOCAL / "doda-word-pairs-v1.json"
if not WORD_PAIRS.exists() or sha256(WORD_PAIRS.read_bytes()).hexdigest() != EXPECTED_DATA:
    prepared = DRIVE_ROOT / WORD_PAIRS.name
    if not prepared.is_file():
        raise FileNotFoundError(f"Put the prepared development artifact at {prepared}")
    content = prepared.read_bytes()
    if sha256(content).hexdigest() != EXPECTED_DATA:
        raise ValueError("The prepared word-pair artifact has the wrong checksum")
    WORD_PAIRS.write_bytes(content)

# Embedded project source snapshot for this notebook.
source = b64decode(
    'UEsDBBQAAAAIAAAAIQAAAAAAAgAAAAAAAAAXAAAAbW9kZWxfcGlsb3QvX19pbml0X18ucHkDAFBLAwQUAAAACAAAACEAlX+8'
    'dk8IAACmGQAAFQAAAG1vZGVsX3BpbG90L2J1bmRsZS5wec1Y3W/jNhJ/918x1ZO01eqaba/oGecD0qyTTbGbDTbetkAuEBhp'
    'FLORSIGkkrhB/vfDkJRE2XG2i3u484ttcr44nI/fMIqi3xQ3CEyUUEtWglkj3B2ArKqaCwQuKlQoCoTrTpQ1ZlEUzWaVkg2s'
    'mV7X/Bp400plQK/Zm7//6La47Fd/3hjUpx9n/u8fWgpH0jITcp8zs+6JFDoSg01b8Rp7mhXSN1Obt1xhYaTa9Byd4IUssWSG'
    'zfo1I1Wx9qY2ssQ6b3ktTUZEvcSj1VG+/P388Ozi9ONZCh8Of89Pz84/r/L3y7OT1Tu3sjr8dLIcl36VBbvuaqY2u7JRkBkq'
    'L9F+92qWbvmtW91lu1FdXpiiJz9R3ZEpZrPDT0fvTlfLo9XnT8sLWMDjDAAgIuotRfndQTSHeKonhTMpMEkd1zUvrde4FKzO'
    'vUbP5xSmU3ck6exp9uHw7PR4ebHK359+OF3BAr5/A6/g4Ls3P8x+W56evFtdDDs/+Y1+/+3pCTEeny7fvyXr4+ge+c3a6NxF'
    'SpRCpGWnCsyLNRa3reTCBHv3UpV5y7gaGJLZbFZiBQpZmV/LTpRYxhRHKdS84SaZ26Pec7O24ZXJFkUcqesoAaZBG4WscTT0'
    'KaQwKAws/E5GcmMrCb6Fg8QS8gpqFLGnTeBfTtUoRTGuEX5ldYdLpaSKq+jRKheswSfAhwKx1DatXAaB5n+ikxI5HQpNp0Rv'
    'jz/kHat5yQzmDRO8Qm3i/oc/Jq9ASANcc6ENEwUOBCmUvOjJnrUx8qb0HNB02sA1AoNfLj6egbz+A4vevEoqqDjWZQr40GJh'
    'sAQufDTa2KqkapjJ71BpLkU0h4N03BS0WfM/mXF7kaiKvGAac41Cc8PvkMIw4GjYQ85F25m8RnFj1tF8NzOn1IapG9win6at'
    'pX/KuMFGx4Fr7sgpsBg8kd2gie1pk4GGV2A2LcaWNgGurePtUu+QBKTysr5ZDG4a1eyJlE7orqWcx7KPjker/ImCnZgoirbN'
    'i5gq1txgYTqFUbInGogzpdC2tlk5lkLApK68FCXPmPeMalfJyB02QJigIIDFVM0lGXA12+vyiKrRwD0eamu9P8IQuFK9dDeO'
    'a+dy3PJXnnzbQHeWvhlw1LCAy6tpxpCpceRiuVgzpamw+WB1/4NQtAtfDsWta7ZcVAC1cRctjZckFTCxiScx+AzzGCO20K2Z'
    'SshNB7RC/zKudcsKjJOJJKnCrpv1WY5xdHZ8FKUwCKIfE87KCybnWEuH3cEKjcYdLLEiesN08sWM4sIWzt1sGhI+uLGMtS2K'
    'MjZdW2OvcLZ7hZNWNppQ8hsKwa+9MccWpKYkyJNVXV03zBTrWEWX373+B3tdXT3++MNTlHpF/+Xht4LSt16Fd9wGdAoRPlDE'
    '5xY4DcX862vl1nEt1/S0dinTRvE2/vKpGq41FzfPn8r3zrAGjQAtfhXeNsGHomZaw89W0rlC6pNSOQuo6eY5F9zkeayxrlIn'
    'NR1DZhMYSxRZbinIJfS9tTeywSKQMRu0tc4Ar4zwTiCfl1QLtiV5hOnaY2xZtl1/AP90GcNLndDv7d7ZF4Zpck/ScsuWZy9m'
    'iDZri2UB14BJwT11CSs6yD3ZmbYz6XAu67XsRiGWm9iGXWZQaKniS17qqySF6WJ/qqtkcmonlmo/Ad4vme2urFWy7AosQUjo'
    'NLuu0Z4gMNZHlhPuQRlNSLmLw7jshxDvqCiK3g8DFCpeceodNP6ABfwMjOo0NZRWSYJXfUBLYSQcnX+2o5WrK14yLOxcFKhy'
    'Dbdvfgs7T2VklY4nqHgU8TdCSD5fiTqioSYE9Unmhog46kz1+qco2e3q00jeD0wt4wir91vkJ4GsNVEKk0FiaPwO8w/AO1vj'
    'gyuDsW0Jvc7L7ani6qW27mnBDhy6a6Dh2tbcvp9P8jl3yGLMfkuiDTNUBV1oku9jP+QOxqbQsDavZWEx7yIq2i5KoTdUinqz'
    'WKkOnbwBU3utmVWQ28qwD9tZEg/zKdvs/+wWN9p5p5fpl15yCReFbFpmuE0BayK4fLOYUQfdwyFKv8mF17oDqXkFsfWcoxxB'
    'mXXYyrETbrO/spptZGfIakdAbaHEcpLF4ACH5yjxjheYkQ5is+4Nt/sNJ6+qJTPfv3lBnl6zdoLbHVj1675neVquK2oRGCcZ'
    'q+s4+StNOXCwkzKHRzsdhgDaRlIe3L2/Y3JHYYJ4cdR4x+p40gK3mlq827x8EbunB5+dKpZOoH3IN/TBV+mOE2HfBO/62WR+'
    'T2ELc4yFc2mRBzBQqI1UWPpE5FQaGQi8D4oiPVX5M1OtHaqhKw9jFUVtuLAZuKeO8mpy5v/peBQyXvXW2Syy/EMSjdIoMF0s'
    'BOX5mxBpvGS7828p0YulImjdSUeqqXVNXEMD1CB3nM4CH2f4wLWh+iPVZJ3rXG+amovb3UJ0zGtcWj5nVsDXF+Sx1/3VB4fJ'
    'HcynYT1ShUPZ3I5PQaXPgt0kYJqMbrtc4Xby//kEYumno+x8DMuAZu/T3Hx/zo/cu49382cKwo62YRqZb9eKgPbZMWVOhdJj'
    'yLxfzXN/C0/jw+DuIzKVhUUYsS1T9q2PafsEbakD4G/YDY0jvqYMFAkBG1cGom3irLktuQom6B6ILAZxU1Q0EPqeyO4wfpxU'
    '31vczMeGaFixjpPMyNg1xx5z2F64mDTC6RhPff0WN2Fb30UhfY8fOJ8GNDOu7UdkdEqH5iaI0JNtA8AJ1HvmsTa2mLfsmlYH'
    'L56aQoLQju2UKaDQncKc6YLzxTGrNSbwLUT/FlHiZ6gB7w464uAuppg5yXzTJKA3wLyBMRwLvIxkJwQUUs+fljgY2/cLeHf2'
    'H1BLAwQUAAAACAAAACEAD/aaVbkFAAAqEAAAEwAAAG1vZGVsX3BpbG90L2RhdGEucHmtV+tu2zYU/q+nONWfSp0qNAV2C6ah'
    'bpJe0C0LWqcoZhgCLR5bXGRSJakkrmFg77A33JMMvOjm1GkL7Jflw3O/fIcMw/BdSSRSoHiNlajXyDVQokkC16Igi6YicpMA'
    '4RQWRBclIC8EZXwFSyFBlwhrQbGCmlVCp2EYBsFSirVVUVREKVTA1rWQuic5jpKosmKL9lSV5On3PwT+319K8Pa74awQFI14'
    '0NK0kEXp9NjPlPO00axSqeS8VVkTmiv82CAv0Lt1TSpGiWaC5wvkRbkm8qpln7ydvn4xOZnm715NjCvBxeQUMngSXJ6/gQyO'
    'gt8nH/LX5xeX0/y3s/OX01eG+JOlTidvX54NyT8GJ9OT/OzDxeT83es/ziGDn4MgeNalIFpK8Ql5NpUNxoElwfsu38cBAADj'
    'daPzoiRSHYNu6gpnSssE0jSdWwZN5AoPcliWZ1b1GnUpqCVQXILJRK4lYZzxVVRUKoGaMKliZ7ezrSADhTqKO7KzeJduesGo'
    'AMadql5Try1tako0RoZhFlZEMx7O4xGj1z/mJJIsWDFklagbyaGoVGSjjpSQGmnk7MRxAiOy1xrHcdDlwCWXURUprJaDyL3u'
    'rcnqMTBO8daGZ78SMGQTJfJmjdI4aeTTQa0SUJpInT2Nd705X6r/zd6w9K3Bo6FBO6aYW7+sSAI3QtJhiamrY+e9ce5OjmeM'
    'qnSFOjKmErg8fxNb91q/BsOZciHXpGKfMArPX5yE3uL8i43/3ADLoOdNM9uhniJXQg6GoUK+0uXnzn2ND54cEA0Ck6yixOIq'
    'N90WDe0kXjYBvK0JV0xwnz+2BC40HMEv2cgz838fJQalJkwhvCdVg2dSChktQysMXng7VLUD0WjFKMLRv3//s93XugvjO55U'
    'yH2vx60jI2C61xMn2Lky0PUZT0ZqW1ck1kg0UtNVzTqqcKkhy0CyValtzxhK4v8zDp9YHbUJdr+zo+N53Cr72DBplQ3D+q6z'
    'YrnINWEVWVQIe3V41JesTVOn8dde7Gsy4uUUbFsNOziZnsBSkjWqBASvNrDtVO567WHc9peoqhbP1HixPkqg0EXeeZuN9oZv'
    'tzAMnzesomajUfSb+PGSSaVB20ZWcMN0CQ33HCcXl76UKnWgMHXRmLklhUapHGoA0XCUwp8oBTBlDZjl3lZ81BXK5ZxTUwSJ'
    'CrlWsKgIv3JJYNwmxl0IRKMt6LcBDMY7GeyR2TyB2Xww4g6W+hTdAaceSsd8A4h1jPcsJbt8ILsfvQ7sKbeOIIPxdurOB2Bi'
    'OtfKx4kX26v2YLO2W3+8Ck2G+sBmpnrzEf46tb11vC2w1vAGN7aNgShA8zFWfrjbu/6ArZVLiVyp2ZP5A7kDsjAlB3fr8teH'
    'QQXC2B1Zuf0rQ0rqGjmNHPq6pu2mn+pNjZk7qgRfxX1e/NXhc9KzrjPuW1A2//ODNjx++r1zGA7CgnDD6EcZCAdc13rjZrFD'
    'QLsz7T6L9kLIhrfRqB0EK53bSbYrMWknML82trOLyWmcjDW1mywbJ8P0mhQ3Lngpbkzszsp8oMFXY+yMJ36rN+PF+hXueDut'
    'P/EXLwan/ZvklGjiytN23jFUTGlLGtzql0KuD1yX+049Hty1PURXgtB88AbKjWNRTXTZY/B7lGy5sY8eIjVbkkJbNKwlLiu7'
    '1RZCl1AQTo07qEBwwGuUm35cDC6kLRwaGxZIdJlKJDRfbDS28MWW/k0UGbY4LfGWshUqHcXwINt/rdzXuuYa9tgiYec2Fahs'
    '46/to86EdH0ETn/or8kWMyGzb7HU5Ec5T2ahPfGQ10XmAFHNQkvxp0MM7gp3GPT2sHOMwMke4iYw3pX7GNin4BtQsI1GYiEk'
    'ha01+dD9ezjfgRZXyFuy/fNwvjv2YLm7C4EeEvY6ORqkwBnM2o9+vva7Ohs9araj5NxddrOwlw/nu3gwuP0gZP0cpONXYfvR'
    'zep/UEsDBBQAAAAIAAAAIQBP+s8zYwQAAEkMAAAeAAAAbW9kZWxfcGlsb3QvZW5jb2Rlcl9kZWNvZGVyLnB5jVbbbuM2EH33'
    'Vwz8RLUKEadFURQVUKO59GF3sQiyQAHDIGhxJLOhSS1JJZs89R/6h/2SgiJ1s53t6sUWdebCM3OGXC6Xv++55aVHC3f3nwB1'
    'aQTaf//+R2D3D56l34NHXu7RQmVsKXUNXAvYmVYLFNABpa7pcrlcLOShMdaDN7bcLyprDvEvpHWtJ4tU6369anXppdFcAXdw'
    'OwfR1kvlqB3hDS8fWcOFQMEcfm5Rl7iIRgcjULFGKuOp4J73Ju/Xf7KH9f3dzQN7d/Ph7uGPHD6urxeLRam4c3ATN34dd020'
    'pu+NaBVmvywAAARWwJjU0jNGHKoqhydT8l2ruH1JmPC4tkFLMjpgs/ETqoqORlBMPMxBntsaPSv33LoZbPZlbrMzAapQkxMX'
    'GXwPqzkaO/RgOP2eLJ18xR6CR5BuUeqm9QwPOxSh+lCA1vSmfyUhk0niET2kc5XDD1c5hApKXTMpvhQf19dHXKVejJ7v7j+R'
    'YPLTjznsuC/3rJLW+eLBtpjDTgppMXVQt5adpfTNdCe7/pbUenHMUltd/Xya25GhaX3T+mj3TmrklnRmk/jZYuy4SEFquI5D'
    'l36ZQl37vZv0XhAFCijOqoMMsLcqSKL/7CjAObpRh0GAzBnrURS3XDkc/I87Zjl4tAcZVF3MakpiriPUom+tToovuSekt9xc'
    'bkc3m9U2y0HIQ3Gxymir3ecW8RXJ5YS0ythnbkUirUt+QtJyubxPseJMu+i2IkCZWnoXrFMxoOwno+vmnd8joBZw4PYRbTft'
    'eqc7rKWGIgZLveaoxmdWtUoRMl8PRSaXWQ6rLB80OFKReovFckAxJaULlM/jJEJWo4e9FAJ1T3nfRNGob6LJ29BKY6t67tHl'
    'wHofKSdyVktknnGWpwROqjtRAIkhJmVTxrk3apZKE3OJ+xh9K75DFT7e0oaLOdU5kMtI8xNXLc6FjFqwxjgZhoY7Kl5PCfWG'
    'xABU4JMscdp0q+MkqCu592gZWeVz9/kwSE9IuaWlNc4x1N6a5oXEzVJvuXaNcRh8XWV5CpGDrLWxyKQWmCZT5/G32CVSV2iD'
    '3lk4B9PpE+itLaJ4+cZJMopkbflOluC8lbp2ORgLH4zGTiZ4aPxLWDFPaKMTaCwK2c1hN1PI+ZY8n8d4FJnHzibCJnpKC/8j'
    'pJQ0FLDZbLuMGUgNlusaj1xk28Gqklq6fTdHN91g28J3MEePWGPBeWxGtye3jHBwTpidqmvOydcl1lFxRlkn8jjW1ylwQynt'
    '7j5bKOBiaJs3kT2xX4H3hUq9y2194F9IGtMzZGDMmuccyr2RJQbiULcHtNxj3CQd5UW9UdJ5kh0RGB5ZDZXaWPO8PUWEpzTa'
    'S93iOfOUQTFecs77mIWBAsIReAJEJavYCr+e3jTP+03t2bmlvGlQi9O72yZleQGr7ZxJWQFXivTZnaFoZ5E/Ho+bzXJJ/zJS'
    'k3QXkxWIIOhwwsULJyqHo8i7tTxipIZX2ZBhGAyxt4v/AFBLAwQUAAAACAAAACEAP1o1fn4DAAC2CAAAFgAAAG1vZGVsX3Bp'
    'bG90L2dydV9jdGMucHmVVe9v2zYQ/a6/4qYvozpFiLNhGIoKmOtmXrHUCzIXKBAEBC2ebC40qZFUfvSvH0hKsmS7A+YviY7v'
    'Hh/v3ZFpmr4XXBisnNCKSah2zLDKoYHl3Wd4Fm4HTAG+NExx5LBYL0C3rmkd7JDxIk3TJKmN3oNwaJzW0oLYN9o42BrdNpvX'
    'JOm+nTbVLmLDvz1OqVGwUKponZC2MEr1iIZVj7RhnCOnFv9pUVWYQ8M49SujYCdlrzlK2gipXcGZYz3PYr2g119u56u/Pv65'
    'yuHT/Atdz++W12t6c71arn/P4Xb+IUne38xXf0AJl0mSVJJZC0vTLlxFlCo+ad5KzN4mAAAca6BUKOEoJRZlncOTrtimlcy8'
    'dhj/s22DhmTFgM0OSyjr4pAE5YhhCnLMbNFR74+dwCYr05ygHj1coiInNBn8ALNphlBN6yjuN8i5UFsoQaniuv8knma0c0QP'
    'XFc5/HgVjPFgKvhLeTv/cHRYVJXmaCLz8u4z8Sk//5TDhrlqR2thrCvXpsUcNuPODLEjrq4TA9WNUMgMmV39kk+NhjeTYhxR'
    'VK6iUlsbSRbrxY22lmwkU49laIQsGbyutXlmhndWh8Pb7i+VqLZuZ0eup2l6h641Kp7sIpwMasP2CFJvhbPAFAdk1Q6eteHf'
    'W3hiUvAOUulWuTBfPWFsdijPzgMZYN/ykkTB2ZHic4VHVWtTIbXaOOTlb0xaHPgPBYxW8hwolBNzSZQ6KrVjDm0EnplbMjAd'
    'azlQdCUrx86TyJsVBu2ONdgdsbDiK5LLLIeLWf4N8020JrIelQTeTDvo0AG+Uzr7g9KR3T1TcG9ginJJABd9x4y+hr4Z89DG'
    '6E2Y2sBZ+JDVtduzF8LFvryYZYUzTNlGWySXOcyyotLKiW2rW0tODjlpdDJs0AuJV8Kx9OnqQWcg/zVe1kLVaLyB1F+53ca+'
    'TluDyF//96DMDduICqwzQm1tDtrASiv0gwe4b9yrj+gnNJEEGoNchPvBTiblP704L2fI3aB1tGExoTOAme2k9loK60Z1jt3o'
    'E+4fhqAX7Xly6NQKBV9FQw4bHAkceEe18b/+yr+fROHcw3Dv9CMquIDZwwnY6wnLfgyF6t9n4qXcv40aHrKTNFHHLPiuhHAh'
    'ThDTbbo6FKxpUHGSpsXfWijSPRCihhm8i29RF3pXnr7CgNJi8P2kkTv+5F9QSwMEFAAAAAgAAAAhAPXZWTFBAQAAYwIAAB8A'
    'AABtb2RlbF9waWxvdC9pbmZlcmVuY2VfaGVscGVyLnB5ZVG7TgQxDOzzFVZo9goi0SJtQUENDbSWSXx7kfKSk73j/h7tJpwO'
    'USWxZ8Yzjtb6rXACgpgdB/hakwsMlBxUljNDEXbeNp9ThXxmgZAtBSi+cDVaa6V8LFkakCyFpPKt0LLYk1JHybGLY/EhNzNG'
    'DFTI5LCXOrRaoWZPhZwpki3XisNDll/Sbk0p5fgIkXyaDs8KAGA3IDDfzJgXWdbIqb3vnclxteLLFmdGdNkiHu6YhpxDGpRJ'
    'd1u6I0iWCvMvcD82aJ16e09rKjdMa8R2EiZXp6ehfgsw3weeNv7YRwc+wEdqQj6x6zurYClB4MU3H6lxuIJwWyVBynDJ4sCt'
    '4tMCF5L4uBbTzci1L+TPcDNuk6aRib8tlwafFFZ+Fclyx6Ja98e+6+mfyEEp5Y+AmCgyIswzaMTtMxB1l+k/o34AUEsDBBQA'
    'AAAIAAAAIQA2NU+17w4AAEosAAAUAAAAbW9kZWxfcGlsb3QvdHJhaW4ucHmVOl1v5DaS7/0ruAICqGdkjT2bSe46EXCTuQQJ'
    'DpsdzDi5B68h0FKpm2eJ1JKU7R6v//uiiqREqWUn54eZllRVLBbru5gkyaXmQjKlmQYzdMCUBNaLVllWcVmLmltgXNYMHnql'
    'LRPWsEZI3jIhG9AgK2A3g6xbyJMk2WxER2Bc73uuDWwarTpWc8urlhsDhgUAU4vKZtOnjGnoW155lAM3h1bcBHBz4G/ffeM+'
    'CRXe/nC0YH75e1j0/4yS4XfH7SH8Vib86ltuG6U7R6jnNl7jY4SiuaxVF54sdH0jWhi3Z5WuDhtHpVM1tCWJLHeSCATff/rw'
    '8y+XP364/O3Tj58z9gN9/KgBd650xlrF69KhZOxeCwv+6ZQwiimQrVTbcgsev4Y7aFXfgbQlQjncO97i0QklyxuQ1aHj+nZi'
    '6/KXn95/uCw///z+7btvMjZCbDabGhrGrepEVRJHKQrJHdN2t2GMsXthD6NE8l95B/UlIGWujz+JFtJa6AKx8p5rkHjG0IKF'
    '4ifeGtgybpgabD9YRw7/bMBnBZ1C6gByyTvYTlD6OKHgn4dyfBKHa5+bdjCHdPHJ5I05yioshFuRKt2uEqhaZWBBYOQ491pL'
    'cppgyETaBb8T0iBbIW/TThgj5L5Ut8WlHmC72WzIFtiHA1S3vRLSGkciSZLLe8W0alsh9xmrEMAMXQc1M62yhinJ7AFYD9oI'
    'Y0FapgfJaqEBle3orBNJ4QmXpZDClmVqoG2yCWo7MYxfcke6YHbo3cE6MPaGNUk18nj2KJ7yGyET1ijNBBMS7WcP6dtIoERv'
    'DxI0qSUr2PnEjwZjlQbPjgFrhdybiJtGDbJmBbu6jl5p2jouNzG7kPhSZfCvUpLkUxB6rgGtED3J4ozhoYLeMtTpX5X9CTn4'
    'UWul1ykKOcDsQy32YGzGyoz1/Ii2yoqwNpqGFSiH9Cb5h0zmC4vGI7O/FN7zpZ7ENj/Ag/uYbnOQlaohTbiphEi2p4z1Wkib'
    'Nskve6m0kHsmZKW6Hs2RTefHHkkQaG1PScbIYLxCLgnSOeS870HW6a9KroCsCsNYjCOFc5w57iT1znvcWcY63petqkg/iqTq'
    'hyRj9yD2B2tKJdvjCk+icaSvkqA0yTVJzT+dikRzYYD9ztsB6DTTSJFHNFaLpgH9XVBMMi2lxZ4i3wilNBsMMM4k3JO5Naqt'
    'QS+OcyYz4nb6Tn4a9doJiHSafgnpdT7skAnDpLIMhT4ZgWjoJZHZLSVDBP6MBKTyjESi0NBxIc13rNdgQN+RDAwwdJWGGNVQ'
    'qTvQx8V2NdhBS+JzfC/hHtW5YB1/SGmpjN3CsWh5d1Nzt8FdOMnJSSTXL/kPR3QOv1lw4WAmR2P43ehl6CQm8cxoL1d7zS4m'
    'TqwG3rEipB+R23DqTYs8vnpFK2QsZnAXLfOUeVIT/uQo3Jd8D/YODypaYxadJ8d3FXH7FXt7nf1/PAd7zcgPsdeBhZcEPz1M'
    'gvWUETiSqT8Fz0nE7JIocjzzxDN2N5vNf42JYtpo9QWkdwYuXn6uDlAPLbiVoVfVweyYkJaeb7itDqURX2B6ZwDq6akFriVG'
    'Yk2K2LSKo7ae5+fnFz4pIqIlkQKTanVvMmb8spn76jeudA2aFawVxqYuELYgCWXrA6KWe1b4JDP/RP+lgViOrLHXnmQAz81h'
    'aJoWUqK+nfYFZgqLN62qblnB/vqWvRqZy6ftb5iPm6ppDNgpUJ9nDFl0tDNHJzrGm6G6BQqYSluoHdyVI7LztF47rOuZYQtZ'
    'w8OOhe1f0fP1VdJyK9C6Jy3ze8nhwaKbnHkUtzy6SG139C97vba96+A/9enWHI1ttoY3cXEqbs+X/+DU2b/zimGxgKL0M2Nc'
    'VwdhobKDhiitijWlhjtRQcZEDdIKe8zYq4zcUgnoTIsL9ArQl63ohC0oxO68xvqoU7DHkeFXr1whNWrPNmNJzEWyWzCVhIWT'
    '3cTDSC8hD5agL9ap82ZleYc5pZJlicT7oz2QIwu1VO7eBKh0G1Fzm0123jFWQ83RpZXufYkZR+p+bynroZ+5PfbAioIlCJ8w'
    'aA0wlw4Q5Sf6dwpVKJEoYZ7SVHdoKFtKvSaQfMo3faa5mbx3x+XA2xKtcG6TDoiqMlbMi7urWMbXV+fXpA/5nar4zdByfdzm'
    'VoWdOifRW9GJL+Qo3Lr0Jn9f8+5/U1oEk0TegQVt0m3GWl2M7Mz8ld+mhZ6SavJnypjSDJ3zYdOrSg2U+LpXovHCiVKLyegd'
    'D1RjUiArnZ4hwlVCH+PwPO7nOYwRIMby8RJsqeXeIQX4qh/wZQz9nIYscv5J1dYJDzV3lIMxRrHOQp+N0svmQvP4CIO4/jEA'
    'L18RVpSQhFT8E7VZasYbC5o9IrUnfwzeryxS8PF0TSlkSVGBFYxCCikZ+R8h99t1p8jO2MWWvXnzbDywyqK64wKUd8wWisKI'
    'i6ouhGAaXHqNi/EpWQ3OC9UKVcoZcCdkGoE6L4csj/DuGO4PogX38ftooemMiY+MNUIb6+IxK1gt7jqF2TWe33wPJzGGFYtg'
    'PhPjaVSPspnOwWAiKyQdQYgP7CzmKItFdEYMRb0BLJIxEE5BaoE67ez1tOiivAs7992g9Gq2iyut7l041Ooe1/F8+gh87To6'
    'sYNaJR6aG54zIfvBmoKecvcQ+bWMWa73MAL4pwhi0V4ZXcYX0Krca16jQy6tKqWSsFLtoWGh7L1nMsYxNgeifBnTL2VMLix0'
    'i7Lel0zYIMyFabARAikhrRTQJ7VSk0glHZJjh1svrUev0BdPySnX+Q2vbu+5rpd9JHJWUuaDFa3Jq1b0JIhSKt2Vq3HgIj/P'
    'GCAzpWjKkZsVcU0CRtYWKztui6iuYXHgeF04QZ5+de7wBNOZFPJiDlB7T8K+OvFcIfhEx7FApRwOXUsRW1F4+1WULBG1lXZM'
    'FOldLXYCgn9Ty2A3JlcZc/5950LBOt7o8ncYXVd2QA5vjCIvEHFBYsdeJkJQz5BxgXjnbSIKu5irTTF3N1OGCGidagi+IXHb'
    'zwLps1ghsp7kexPun8v2MGycLvJ0YshzkT3f//qRdO/RqSCa6JvHRVx7Yh1w7yadYT+OtvAmOoZd/q75oybZM/nDORruXPmh'
    '9RGTfcUuzs+f0eewix8mP/P05jEKpk+ZZ5lM9jkWffVCmpLFRpL5vCei6GsbP0aA1CO5KqfnQptZeTMRK0PbM9Q5UZnldoaN'
    'corYUzM5mcYVOQ5xkpCe0hgBHoSxJo2880lbVwNNNgoaAVEGaqgh71oKFh7ssrcvGo90FfX/SteoSK7xIE72tBIdnEQDIVMp'
    'DSbK+XwHOZ3iR8b+B47+1+Wxdy8XcafnxkylRg53vPW+u3ezI6EkVYFPoZ7v8NEF9GnuQ+//qOInoO3KKTHyo4PEdgIBnZb7'
    'U909bZjaStTQdB7AgjRKL6sh334qKYVI75Wut1viEn8ij7Ry1OWkVGOslMZwqaXMe6w24J8DTiJTv37YD2VSCyNtQe7tYSLm'
    'ObxCaYjaOD5EbZANTy4uc2gkNCUhew1QH9OVXMivE/X2psPLh56siraeeaKTBMIbZOGL6FOSRnhpyHcGgKlw81UgqSArpple'
    '6q01Wt6Brkz5UjKfeuh6kz6u2MVuzdCD2u/84k8ZZbfSFm+xJKHxRug3bmeeyMF7V2PUoCsoQ8vPaaFWyoaZYFli67kst1i9'
    'q/YO0q2fMJqrC6crfmoyDk1Swp/5l2kcmvfHZDvvOY7WQj4K50qu3zXSiUayyTbft+omTV4RochsHBfhjMkNYaNjved6jj3X'
    'sVMbPNbYBA3imMnNLzBvkKIMsVJIRzerDXUXwig+f6/3A46JP9KXtAZTadHTsKUsa1WV5TbCzHldl9yjpAnqZUmqlGQMQ3bx'
    'cRx4riLoQf5JyLOzWcMKo4kSFZhiMT/X8M9BaKgjg36GngvpYXnhZtANH1rUyZcQyW2coU9bRb54+x8vomOXaBXx65eXxST1'
    'jDLa9WVfRsZ4fWYPwpwZMNiCi4h4U9fUNvQE6D8kEQaeoqFyFt/4bCgjlKhT4F9MqfeWfc8uMCmfWrUOgtJ95Kb03MROim6S'
    'PAf3va8oIlPyHIMbUwXeXJLm2IomVkJa0He8zWiVQNU1JrrBWHYDrFdGWHEHybjzFHmL0lVhSn7HRctvWki3uMP13mXFe34j'
    'WmGP6fkWR47ptxl7tyjANEsuv05o+0K+1AQ93+ambwVa+nO75+zDb//9/gwkclazy69RsMEmvsMxDVSWBpWXX+Mw0oqOxogf'
    'VMtvkpBAuEzy0rcJ8NbA44tsPbn7Hh+PlwgVgKOe8FqmSbdVivX7KU7NJncSPNudMH70Ng8EowLnepB5d1sLnXq3TytmjPLD'
    '6Q4FMUB7GGO8e0ypwtidJ1Ej9/k8GJGpsz/5dd/i35t83lIP3GXjHOpP2RK2lKciJ+TLTfIYhPG0e1zc1UFhR7OChU1OxKIJ'
    'wjPW5mBHK/D9Ntp5pIBeXT5b1ffYs7Rxp3JeiTzl7BPgDNwqf5Usf6ZMclFs0cQvx9gdV+8zJSB1mef4C12aB8+rxMWr60j1'
    'owTnxdpmOtIXSpvFuW6jxCceKGBPSw5daQ/InkkvgsqH+3PF8mqY7/tU/YCjWsr/n2nY1WCskGEwG7im+znuKtnZ48kGrnYX'
    '31w/jTUWOqeIzEq5NR5NfEdtPqKLCKyayIL5zFvfvCPlDP8k7SxWjmA68wC0vNW2RjoYVhF+TFBOnNh8WNl3SCdjaWM2yKVo'
    'MBOjmnX7wvyaaNMFSiqOoot/MU2fwg9VBcY0A7ogujM3pqWhPEoTw1veJRlLLr/Ff+23CXvF/jNjyS2X1UENDb692R+ExR9/'
    'bXl1SGZzeTO01pxeqOqDCtIyo4pmI/eLWnX1gpWnHq68jDRz/8tXfPNOiCuVp0r5D8nObx9RPU/fr87pBlB4urhetApOO7r+'
    '1mbPtbBH1nCB8RWl0YgH/CUeUInZI7L9Fx23d6Oz+lcRMxDlPLGdTfDRWZxcx1nw45d3bhNqJhUbDCYBcU23CPC/gxaNQE9N'
    'tHbsMdKzp3/Iv3nVZZ9/fn/29t03CEDKugznm43A64KYDZQltenKEsuMsvRjN1dzbP4NUEsDBBQAAAAIAAAAIQD/PNsa/wUA'
    'AJcOAAAXAAAAdmFsaWRhdGlvbl9iZW5jaG1hcmsucHmNV81uGzcQvuspJjztohtVdmwnFrCHID9tCjcJ6qAXV1hQXK7EZJdc'
    'kJRr1dChaJMUeYu2KIz2UhTopW8ivU0xJFfi2m5QXURxvvnlN0OKEHLKlOZQiQtewrdKl9BqXgpmhZIGlAQ753C+B49fPKZw'
    'TmtRUhSBYmyhNZeMmyEhZDAQTau0BapnLdWGDyqtGmCqrnmwFQCP1EJarjMoeUUXtUVfHjynZl6LaQc0c7p/eNTZfW2U9LCW'
    '2hj2ktp5B1pIwVTJS2rpYPDwq1fPnj589Ko4/fzh/uER5EAeHOxP6f7B0agalQf7e8dsejyiB3uM0REr2ej+cbn3oCr39g6O'
    'yqP7h0f0+PBo/8G9e+y42q+m1XRKBi+fnbx4VZy+fHJy8uz5Z6eQw+UAAIA09ygZwyXZvF3/QzJw3+srssq8+A1dygD4YX21'
    '+bB5t/7dwbpfO2jlYd878febD1uBaKw3sf5t83b9x/oKEeursO5QlM2l8rBf139v3m1+8rBo/UtYdypzsQyxvd98cG7fbz5E'
    'Jusm+L3a/Lh56/3uVt3u+uedwpyWOpj8cfN+/ef6r63W+/VV+L3KBqvB6cmzR0+wjglhqmmURJg655rWNS4X0nAuSToYDEpe'
    'gVS6obX4jhdU06lgieUXNh07t5rbhZZAyPC1EjJxe/hhc6qhUtovhIxpMtwaTMjzp49IBs7gVldUPTSjls+UXiZoKoU7OZAv'
    'JXHoLsIpl2zeUP0maanQJov7KX+uJA/RWk2FFHIGedwJSWgPHwEGjVYwaGftjDg1MhlvI+zsnCHgjNTUonwSfvoqkckEPslh'
    'zyn5MhdKl1xDDkZpy8ukM5PBG77Ma9pMS+rmwRiSu1sfuDEZWmVpnaSZk6fp2fhwNJpEttEqt0nsyOcjqt50EQawILtcYmHX'
    'Wd3Hx9IImTAskekFqnmFcXrJmebVJMO9NO2ZwHKimQw8EMvapTYUljcm2SmsBoNbQvJR3GChWth2YdPIg99xB7ez0DlZhWK5'
    'IOJEJW34GBwRogGbjzIwVgtmC6a05szizjnVgsreViOMEXKWj3ZpYERoFSPxvTbw2fkIzrnmJYbgPeOxpbforP6LjrsbIeak'
    'L4HmFXcJQA49cmbQJ+dWLxQt79Vsxm3iiLaF+VpAvsXnO19bEK0t15Jacc6xxtcG99ZqBknEklBTJLD3oTQku6Ps2aSyjFK8'
    'Jo2CrQVzEZxtx9okHi/uzhUytM64R1ivO6Rty2W5HZDpDXWpbMzljxrZDtTbGOKxfX3NzaLGkoTuQuzkFsQZiShLooFzHdan'
    'skf6vVvh13ju8WHzVoXQBR7YHZ4fNT18YL9PaEjLMtBsy3UUZF3+7oiwAF0T76rUOS6FsUIyW1RKN1iCHGqO8yry0697wzFv'
    'tJ2EsuCdF5Ijaf8g3NiF/PZy31aJilx6B6uCIpaypQvqpnhX20+DG1GFBa8N97WLL9nLQMaQ6jgkiltpBsTziIxDyVbhbtSc'
    'lkXU2wm+5UKWLV3WiuIswqfeENdePnRa06XlpmtVUXnKGyGNpZLxJGhnbnam2LZULpNroHNaL7gbpX7Gud9+mDntodtAL9HZ'
    'UmE4fI2CJ1ornZD4RmgWxsKUA4UvTl88BzV9zZmFhrYt3uwnOO7AVQischyXMxNaLxQyuA4FaqiQybYg2rgbuntSDx/q2aLh'
    '0r50kqTkhmnRYiR5UZSKFUUaaSKjCxpUEoLcLtzUxkfOsuU5vpw/onD3bpRprJPBnNdtTv5Hyr77upSpnhl3Fzhv7gv9dbeu'
    '1ctd3fG55XM3w13oPS7EY9D/WUhQKx3O+UUpZtzgXXYnh2t/BK5Ntxvni97uumuOaisqyiyUihvHuIZaNu/+Enkf0STtvxVu'
    'kN3lEm2kGPj1zajfdt3s/tzk0dMyahGX8hnxJzvpPTd9ZPyC8dZC8uLUJZhFyaZADXBcRW8wfzr8QthkL4OKeDlcuu/VN93V'
    '0WohrQ+kXDStSXyYGQhZcmnz/cw9LYs3fGnyV3rBU3whiwqKAgdhUeC1TYoCKV8UxAfg+T/4F1BLAQIUAxQAAAAIAAAAIQAA'
    'AAAAAgAAAAAAAAAXAAAAAAAAAAAAAACAAQAAAABtb2RlbF9waWxvdC9fX2luaXRfXy5weVBLAQIUAxQAAAAIAAAAIQCVf7x2'
    'TwgAAKYZAAAVAAAAAAAAAAAAAACAATcAAABtb2RlbF9waWxvdC9idW5kbGUucHlQSwECFAMUAAAACAAAACEAD/aaVbkFAAAq'
    'EAAAEwAAAAAAAAAAAAAAgAG5CAAAbW9kZWxfcGlsb3QvZGF0YS5weVBLAQIUAxQAAAAIAAAAIQBP+s8zYwQAAEkMAAAeAAAA'
    'AAAAAAAAAACAAaMOAABtb2RlbF9waWxvdC9lbmNvZGVyX2RlY29kZXIucHlQSwECFAMUAAAACAAAACEAP1o1fn4DAAC2CAAA'
    'FgAAAAAAAAAAAAAAgAFCEwAAbW9kZWxfcGlsb3QvZ3J1X2N0Yy5weVBLAQIUAxQAAAAIAAAAIQD12VkxQQEAAGMCAAAfAAAA'
    'AAAAAAAAAACAAfQWAABtb2RlbF9waWxvdC9pbmZlcmVuY2VfaGVscGVyLnB5UEsBAhQDFAAAAAgAAAAhADY1T7XvDgAASiwA'
    'ABQAAAAAAAAAAAAAAIABchgAAG1vZGVsX3BpbG90L3RyYWluLnB5UEsBAhQDFAAAAAgAAAAhAP882xr/BQAAlw4AABcAAAAA'
    'AAAAAAAAAIABkycAAHZhbGlkYXRpb25fYmVuY2htYXJrLnB5UEsFBgAAAAAIAAgALQIAAMctAAAAAA=='
)
assert sha256(source).hexdigest() == "c51ea9fd5b77067d674664147d9c45ec0f8a15fc4055e6ad2945c600e056d564"
with ZipFile(BytesIO(source)) as archive:
    for member in archive.namelist():
        path = LOCAL / member
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(archive.read(member))


In [ ]:
import os
import subprocess
import sys

for architecture in CANDIDATES:
    run = DRIVE_ROOT / "runs" / RUN_NAME / architecture
    command = [
        sys.executable, "-m", "model_pilot.train", str(WORD_PAIRS), str(run),
        "--architecture", architecture, "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE), "--seed", str(SEED), "--save-every", str(SAVE_EVERY),
    ]
    if STEPS_THIS_SESSION is not None:
        command.extend(["--steps-this-session", str(STEPS_THIS_SESSION)])
    subprocess.run(command, cwd=LOCAL, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"}, check=True)


## Results

Look under `3aransiya/runs/<RUN_NAME>/<architecture>/` in Drive. Each finished candidate writes `validation.json` and prints the verified bundle directory and manifest SHA-256. Download the complete `bundle-…` directory for Fedora.

Keep the same code, Python, PyTorch, GPU type, and training settings when resuming. A changed runtime or configuration stops with an error to prevent a silent continuation under different assumptions. The two architectures have independent checkpoints.
